# Output Parsers in LangChain

## 1. What are Output Parsers?

Output Parsers in LangChain are components used to **convert raw LLM responses into structured and usable formats**.

An LLM normally returns a response as text. However, in real applications we often need the response in a specific structure such as:

- String
- JSON
- Dictionary
- List
- Structured fields
- Pydantic objects
- Custom application-specific formats

Output parsers help us take the raw LLM output and convert it into the required format.

### Basic Flow

    User Input
        ↓
    Prompt
        ↓
    LLM
        ↓
    Raw LLM Response
        ↓
    Output Parser
        ↓
    Structured / Usable Output

### Example

Raw LLM output:

    Name: Musharraf
    Age: 24
    City: Mumbai

After parsing:

    {
        "name": "Musharraf",
        "age": 24,
        "city": "Mumbai"
    }

This makes the output much easier to use inside Python programs and applications.

---

# 2. Why Do We Need Output Parsers?

LLMs are probabilistic systems. Even when we give instructions, the response may not always follow exactly the format we expect.

For example, suppose we ask:

    Give me the name, age and city of the person in JSON format.

The model may return:

    Sure! Here is the information:

    {
        "name": "John",
        "age": 25,
        "city": "Mumbai"
    }

The extra sentence can cause problems if our application expects only JSON.

Output parsers help us:

- Convert raw text into structured data.
- Maintain consistency in application output.
- Validate the output.
- Extract required fields.
- Convert model responses into Python objects.
- Make LLM outputs easier to use programmatically.
- Reduce manual string processing.
- Integrate LLM outputs with databases, APIs and business logic.

---

# 3. Important Output Parsers in LangChain

Common output parsers include:

1. StrOutputParser
2. JsonOutputParser
3. StructuredOutputParser
4. PydanticOutputParser

They solve different problems.

---

# 4. StrOutputParser

## Definition

`StrOutputParser` is the **simplest output parser in LangChain**.

It is used when we want the final LLM response as a **plain Python string**.

### Flow

    LLM
      ↓
    AIMessage
      ↓
    StrOutputParser
      ↓
    String

The LLM response may internally contain metadata such as:

- token usage
- model name
- finish reason
- response metadata
- other information

`StrOutputParser` extracts the actual textual content.

---

## Why use StrOutputParser?

Use it when:

- You only need text.
- You don't need JSON.
- You don't need field validation.
- You don't need a specific schema.
- You want a simple string response.

---

## Example

    from langchain_core.output_parsers import StrOutputParser

    parser = StrOutputParser()

    result = parser.invoke(
        AIMessage(content="A black hole is a region of space.")
    )

    print(result)

Output:

    A black hole is a region of space.

The result is a normal Python string.

---

## Using StrOutputParser with an LLM

    from langchain_openai import ChatOpenAI
    from langchain_core.output_parsers import StrOutputParser

    model = ChatOpenAI()

    parser = StrOutputParser()

    chain = model | parser

    result = chain.invoke("Explain artificial intelligence in one sentence.")

    print(result)

The parser converts the model's message into a simple string.

---

# 5. JsonOutputParser

## Definition

`JsonOutputParser` is used when we want the LLM to return **JSON-formatted structured data**.

### Flow

    LLM
      ↓
    JSON response
      ↓
    JsonOutputParser
      ↓
    Python dictionary / structured JSON data

---

## Why use JsonOutputParser?

Use it when:

- You need JSON output.
- You want structured data.
- You don't necessarily need a Pydantic object.
- You want to integrate the result with APIs or databases.
- You want a lightweight structured output approach.

---

## Example

Suppose we want:

    {
        "name": "Musharraf",
        "age": 24,
        "city": "Mumbai"
    }

The model can be instructed to return JSON.

Example:

    from langchain_core.output_parsers import JsonOutputParser

    parser = JsonOutputParser()

    result = parser.invoke(
        '{"name": "Musharraf", "age": 24, "city": "Mumbai"}'
    )

    print(result)

The parsed result can be used like a Python dictionary.

---

## Accessing Values

    print(result["name"])
    print(result["age"])
    print(result["city"])

Output:

    Musharraf
    24
    Mumbai

---

# 6. JSON vs Python Dictionary

JSON is a **data interchange format**.

Python dictionary is a **Python data structure**.

Example JSON:

    {
        "name": "Musharraf",
        "age": 24
    }

After parsing, Python can represent it as:

    {
        "name": "Musharraf",
        "age": 24
    }

The syntax looks similar, but conceptually they are different.

### JSON

Used mainly for:

- APIs
- Data exchange
- Storage
- Communication between applications

### Python Dictionary

Used mainly inside Python programs.

---

# 7. StructuredOutputParser

## Definition

`StructuredOutputParser` is used when we want the LLM response to follow a **predefined list of fields**.

It allows us to define the fields that the model should return.

### Basic Idea

    Define fields
        ↓
    Generate format instructions
        ↓
    Send instructions to LLM
        ↓
    LLM generates structured response
        ↓
    StructuredOutputParser
        ↓
    Parsed structured data

---

## Why use StructuredOutputParser?

It is useful when:

- You know the fields you want.
- You want a predictable output structure.
- You want format instructions generated automatically.
- You don't necessarily need full Pydantic validation.

---

## Important Concept: ResponseSchema

`ResponseSchema` is used to describe the expected fields.

Example:

    from langchain.output_parsers import (
        StructuredOutputParser,
        ResponseSchema
    )

    response_schemas = [
        ResponseSchema(
            name="name",
            description="Name of the person"
        ),
        ResponseSchema(
            name="age",
            description="Age of the person"
        ),
        ResponseSchema(
            name="city",
            description="City where the person lives"
        )
    ]

Then create the parser:

    parser = StructuredOutputParser.from_response_schemas(
        response_schemas
    )

---

## Format Instructions

The parser can generate instructions that tell the LLM how the output should be formatted.

Example conceptually:

    format_instructions = parser.get_format_instructions()

These instructions can be inserted into the prompt.

Example:

    prompt = f"""
    Give information about a person.

    {format_instructions}
    """

The LLM is then guided to return the expected structure.

---

## Example

    from langchain.output_parsers import (
        StructuredOutputParser,
        ResponseSchema
    )

    response_schemas = [
        ResponseSchema(
            name="name",
            description="Name of the person"
        ),
        ResponseSchema(
            name="age",
            description="Age of the person"
        ),
        ResponseSchema(
            name="city",
            description="City of the person"
        )
    ]

    parser = StructuredOutputParser.from_response_schemas(
        response_schemas
    )

    format_instructions = parser.get_format_instructions()

    print(format_instructions)

The format instructions tell the model what fields to return and how the output should be structured.

---

# 8. PydanticOutputParser

## Definition

`PydanticOutputParser` is a structured output parser that uses **Pydantic models** to define and validate the expected output.

It is more powerful than a basic structured parser because Pydantic provides:

- Schema definition
- Type validation
- Data validation
- Default values
- Type conversion
- Constraints
- Descriptions
- Better reliability for application code

---

## Basic Flow

    Pydantic Model
          ↓
    PydanticOutputParser
          ↓
    Format Instructions
          ↓
    Prompt
          ↓
    LLM
          ↓
    Structured Response
          ↓
    Pydantic Object

---

# 9. Pydantic

## Definition

Pydantic is a Python library used for:

- Data validation
- Data parsing
- Schema definition
- Type enforcement

It helps ensure that data is:

- Correct
- Structured
- Type-safe
- Valid according to the defined model

---

## Basic Pydantic Example

    from pydantic import BaseModel

    class Person(BaseModel):
        name: str
        age: int
        city: str

Now we have a schema:

    Person
    ├── name -> str
    ├── age  -> int
    └── city -> str

---

## Creating an Object

    person = Person(
        name="Musharraf",
        age=24,
        city="Mumbai"
    )

    print(person)

The data follows the schema defined by the model.

---

# 10. PydanticOutputParser Example

    from pydantic import BaseModel, Field
    from langchain_core.output_parsers import PydanticOutputParser

    class Person(BaseModel):
        name: str
        age: int
        city: str

    parser = PydanticOutputParser(
        pydantic_object=Person
    )

Now the parser knows that the LLM should return:

    name -> string
    age  -> integer
    city -> string

---

# 11. Field in Pydantic

Pydantic's `Field()` can provide additional information.

Example:

    from pydantic import BaseModel, Field

    class Person(BaseModel):
        name: str = Field(
            description="Name of the person"
        )

        age: int = Field(
            description="Age of the person"
        )

        city: str = Field(
            description="City of the person"
        )

Descriptions help explain the expected fields.

---

# 12. Default Values in Pydantic

Pydantic allows default values.

Example:

    from pydantic import BaseModel, Field

    class Person(BaseModel):
        name: str
        age: int = 18
        city: str = "Unknown"

If some values are not provided, defaults can be used depending on the model configuration and validation rules.

---

# 13. Optional Fields

Some fields may not always be available.

Example:

    from typing import Optional
    from pydantic import BaseModel

    class Person(BaseModel):
        name: str
        age: int
        email: Optional[str] = None

Here:

    email

can be absent or `None`.

---

# 14. Validation with Pydantic

Pydantic can enforce constraints.

Example:

    from pydantic import BaseModel, Field

    class Person(BaseModel):
        name: str
        age: int = Field(gt=0)

Here:

    gt=0

means:

    age > 0

If invalid data is supplied, validation can fail.

---

# 15. Type Conversion

Pydantic can perform appropriate type parsing/conversion in many cases.

Example:

    class Person(BaseModel):
        age: int

Input:

    age = "24"

Pydantic may parse this into:

    age = 24

depending on the type and validation rules.

This is one advantage of Pydantic over simple type hints.

---

# 16. PydanticOutputParser with Format Instructions

One important method is:

    parser.get_format_instructions()

It generates instructions that can be included in the prompt.

Example:

    format_instructions = parser.get_format_instructions()

    prompt = f"""
    Extract the person's information.

    {format_instructions}
    """

The LLM receives instructions about the expected structure.

---

# 17. Complete PydanticOutputParser Example

    from pydantic import BaseModel, Field
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import PydanticOutputParser

    class Person(BaseModel):
        name: str = Field(
            description="Name of the person"
        )
        age: int = Field(
            description="Age of the person"
        )
        city: str = Field(
            description="City of the person"
        )

    parser = PydanticOutputParser(
        pydantic_object=Person
    )

    prompt = PromptTemplate(
        template="""
        Extract information about the person.

        {format_instructions}

        Person:
        {text}
        """,
        input_variables=["text"],
        partial_variables={
            "format_instructions":
                parser.get_format_instructions()
        }
    )

    model = ChatOpenAI()

    chain = prompt | model | parser

    result = chain.invoke(
        {
            "text":
            "Musharraf is 24 years old and lives in Mumbai."
        }
    )

    print(result)

The final result is a Pydantic object.

---

# 18. TypedDict

`TypedDict` is a way to define the expected structure of a Python dictionary using type hints.

Example:

    from typing import TypedDict

    class Person(TypedDict):
        name: str
        age: int
        city: str

This tells developers and type checkers that a dictionary should contain:

    name -> str
    age  -> int
    city -> str

---

# 19. Why Use TypedDict?

TypedDict is useful when:

- You only need type hints.
- You want to describe dictionary structure.
- You don't need runtime validation.
- You trust the data source.
- You want lightweight type information.

Important:

    TypedDict does NOT provide runtime data validation
    like Pydantic does.

---

# 20. TypedDict vs Pydantic

## TypedDict

TypedDict mainly provides:

- Type hints
- Dictionary structure
- Better IDE support
- Static type checking

It does not normally validate the actual data at runtime.

Example:

    class Person(TypedDict):
        name: str
        age: int

This describes the expected structure.

---

## Pydantic

Pydantic provides:

- Type validation
- Runtime validation
- Data parsing
- Default values
- Constraints
- Type conversion
- Schema generation

Example:

    class Person(BaseModel):
        name: str
        age: int

Pydantic can validate actual input data.

---

# 21. TypedDict vs Pydantic Example

TypedDict:

    from typing import TypedDict

    class Person(TypedDict):
        name: str
        age: int

Pydantic:

    from pydantic import BaseModel

    class Person(BaseModel):
        name: str
        age: int

Main difference:

    TypedDict
        ↓
    Type hint / structure

    Pydantic
        ↓
    Structure + validation + parsing

---

# 22. JSON Schema

JSON Schema is a standard way to describe the expected structure of JSON data.

Example:

    {
        "type": "object",
        "properties": {
            "name": {
                "type": "string"
            },
            "age": {
                "type": "integer"
            },
            "city": {
                "type": "string"
            }
        },
        "required": [
            "name",
            "age",
            "city"
        ]
    }

This schema tells us:

    The output must be an object.

    It should contain:
        name -> string
        age  -> integer
        city -> string

---

# 23. Why Use JSON Schema?

JSON Schema is useful when:

- You want a standard JSON structure.
- You need schema-based validation.
- You don't want to depend on Pydantic.
- You need interoperability between different programming languages.
- You want a schema that can be understood by APIs and other systems.

---

# 24. TypedDict vs Pydantic vs JSON Schema

## TypedDict

Best for:

    Basic structure + type hints

Characteristics:

    Lightweight
    Python-specific
    Mainly static typing
    No strong runtime validation

---

## Pydantic

Best for:

    Validation + parsing + Python objects

Characteristics:

    Runtime validation
    Type enforcement
    Default values
    Constraints
    Python-friendly

---

## JSON Schema

Best for:

    Standardized JSON structure and validation

Characteristics:

    Language-independent
    Standard JSON-based schema
    Useful for APIs
    Can define validation rules

---

# 25. When to Use What?

## Use TypedDict if:

- You only need type hints.
- You need basic structure enforcement.
- You don't need runtime validation.
- You trust the LLM output.
- You want a lightweight solution.

Example:

    class Person(TypedDict):
        name: str
        age: int

---

## Use Pydantic if:

- You need data validation.
- You need default values.
- You need constraints.
- You want automatic type parsing.
- You need Python objects.
- You are building a production application.
- The output must follow a strict schema.

Example:

    class Person(BaseModel):
        name: str
        age: int = Field(gt=0)

---

## Use JSON Schema if:

- You want a standard JSON format.
- You need schema-based validation.
- You don't want to depend on Python/Pydantic.
- Multiple systems or languages need to understand the schema.
- You need JSON interoperability.

---

# 26. Comparison Table

| Feature | TypedDict | Pydantic | JSON Schema |
|---|---|---|---|
| Basic structure | Yes | Yes | Yes |
| Type hints | Yes | Yes | Yes |
| Runtime validation | No | Yes | Yes |
| Default values | Limited / type-level | Yes | Yes |
| Constraints | No | Yes | Yes |
| Automatic parsing | No | Yes | Depends on implementation |
| Python object | Dictionary | Pydantic object | JSON |
| Language independent | No | Mostly Python-focused | Yes |
| Lightweight | Yes | Medium | Yes |
| Best for | Type hints | Validation + parsing | Standard JSON schema |

---

# 27. Output Parser Comparison

| Parser | Main Purpose | Output |
|---|---|---|
| StrOutputParser | Extract plain text | String |
| JsonOutputParser | Parse JSON | Dictionary / JSON-compatible data |
| StructuredOutputParser | Predefined fields | Structured data |
| PydanticOutputParser | Structured + validated output | Pydantic object |

---

# 28. StrOutputParser vs JsonOutputParser

## StrOutputParser

Input:

    LLM response

Output:

    "Musharraf is a student."

Use when:

    You only need text.

---

## JsonOutputParser

Input:

    {
        "name": "Musharraf",
        "age": 24
    }

Output:

    Python dictionary / parsed JSON data

Use when:

    You need structured JSON.

---

# 29. JsonOutputParser vs PydanticOutputParser

## JsonOutputParser

Good when:

    You need JSON data
    +
    You don't require a Python model

Example:

    {
        "name": "Musharraf",
        "age": 24
    }

---

## PydanticOutputParser

Good when:

    You need structured data
    +
    Type validation
    +
    Constraints
    +
    Python model

Example:

    Person(
        name="Musharraf",
        age=24
    )

---

# 30. StructuredOutputParser vs PydanticOutputParser

## StructuredOutputParser

Uses:

    ResponseSchema

Main goal:

    Tell the LLM which fields to return.

Good for:

    Simple structured responses.

---

## PydanticOutputParser

Uses:

    Pydantic BaseModel

Main goal:

    Define + validate + parse structured data.

Good for:

    More robust applications.

---

# 31. Important Concept: LLM Output Is Not Automatically Reliable

Suppose we expect:

    {
        "name": "John",
        "age": 25
    }

The model might return:

    Here is the information:

    {
        "name": "John",
        "age": "25"
    }

or:

    {
        "name": "John"
    }

or:

    The person is John and he is 25 years old.

Therefore, structured output techniques are important when application code depends on a predictable schema.

---

# 32. Output Parser and Prompt

Output parsers often work together with prompts.

General pattern:

    parser
       ↓
    format instructions
       ↓
    prompt
       ↓
    LLM
       ↓
    parser
       ↓
    final output

Example:

    format_instructions = parser.get_format_instructions()

    prompt = f"""
    Extract the information.

    {format_instructions}
    """

This gives the LLM instructions about the expected output.

---

# 33. Important Methods to Remember

## StrOutputParser

Main purpose:

    Convert AIMessage / model output into string.

---

## JsonOutputParser

Main purpose:

    Parse JSON-formatted output.

---

## StructuredOutputParser

Important concepts:

    ResponseSchema
    from_response_schemas()
    get_format_instructions()

---

## PydanticOutputParser

Important concepts:

    Pydantic BaseModel
    pydantic_object
    get_format_instructions()
    validation
    parsing

---

# 34. Practical Example: Movie Information

Suppose we ask an LLM:

    Give information about the movie Interstellar.

We want:

    title
    director
    year
    genre

Without structured output:

    Interstellar is a science-fiction movie directed by
    Christopher Nolan and released in 2014.

This is difficult to process programmatically.

With structured output:

    {
        "title": "Interstellar",
        "director": "Christopher Nolan",
        "year": 2014,
        "genre": "Science Fiction"
    }

Now our application can easily do:

    result["title"]

    result["director"]

    result["year"]

    result["genre"]

---

# 35. Practical Example: Sentiment Analysis

Suppose an application needs:

    sentiment
    confidence
    explanation

Expected output:

    {
        "sentiment": "positive",
        "confidence": 0.94,
        "explanation": "The user expressed satisfaction."
    }

Pydantic can define:

    class Sentiment(BaseModel):
        sentiment: str
        confidence: float
        explanation: str

We can additionally restrict sentiment values.

Example concept:

    sentiment:
        positive
        neutral
        negative

This makes the application more reliable.

---

# 36. Practical Example: Dataset Generation

Suppose we want to generate a dataset using an LLM.

Required fields:

    question
    answer
    difficulty
    topic

Expected output:

    {
        "question": "What is a neural network?",
        "answer": "A computational model...",
        "difficulty": "medium",
        "topic": "Deep Learning"
    }

Using structured output allows every generated record to follow the same schema.

This is extremely useful for:

- Dataset generation
- Synthetic data
- Fine-tuning datasets
- Evaluation datasets
- RAG pipelines
- Classification systems

---

# 37. Practical Example: API Integration

Suppose an LLM analyzes a customer message.

Expected:

    {
        "intent": "refund",
        "priority": "high",
        "customer_name": "John"
    }

The application can then automatically call another API:

    LLM
      ↓
    Output Parser
      ↓
    Structured Data
      ↓
    Business Logic
      ↓
    API / Database

This is one of the major reasons structured output is important in production GenAI systems.

---

# 38. Output Parsing in a LangChain Chain

General LangChain pattern:

    chain = prompt | model | parser

Example:

    chain = prompt | model | StrOutputParser()

or:

    chain = prompt | model | JsonOutputParser()

or:

    chain = prompt | model | PydanticOutputParser(...)

This is an important LangChain pattern.

---

# 39. LCEL Perspective

LangChain Expression Language (LCEL) allows components to be composed using:

    |

For example:

    prompt | model | parser

means:

    prompt output
        ↓
    model input

    model output
        ↓
    parser input

    parser output
        ↓
    final result

This makes chains concise and readable.

---

# 40. Important Difference: Parsing vs Validation

These concepts are related but not identical.

## Parsing

Parsing means:

    Convert data from one representation into another.

Example:

    JSON string
        ↓
    Python dictionary

---

## Validation

Validation means:

    Check whether the data follows the required rules.

Example:

    age must be an integer
    age must be greater than 0
    sentiment must be positive/neutral/negative

Pydantic is particularly strong because it combines:

    Parsing + Validation

---

# 41. TypedDict Important Limitation

Remember:

    TypedDict is mainly for type checking.

It does not behave like a validation model.

Example:

    class Person(TypedDict):
        name: str
        age: int

This does not automatically mean that every dictionary created in Python will be runtime-validated.

Therefore:

    TypedDict != Pydantic

---

# 42. Pydantic Important Advantage

Pydantic provides a real model.

Example:

    class Person(BaseModel):
        name: str
        age: int

Then:

    person = Person(
        name="John",
        age=25
    )

The object is validated according to the schema.

It can also be converted into a dictionary:

    person.model_dump()

And into JSON:

    person.model_dump_json()

---

# 43. Important Pydantic Concepts to Remember

### BaseModel

Used to create Pydantic models.

    class Person(BaseModel):
        name: str
        age: int

### Field

Used for:

- Description
- Defaults
- Constraints
- Metadata

Example:

    age: int = Field(
        gt=0,
        description="Age of the person"
    )

### Validation

Checks whether input follows the model.

### Parsing

Converts input into the expected model structure.

### model_dump()

Converts the model into a Python dictionary.

### model_dump_json()

Converts the model into JSON.

---

# 44. Common Output Parser Decision Tree

Ask:

    Do I only need text?
        ↓
       YES
        ↓
    StrOutputParser

If not:

    Do I need JSON?
        ↓
       YES
        ↓
    JsonOutputParser

If not:

    Do I need predefined fields?
        ↓
       YES
        ↓
    StructuredOutputParser

If I additionally need:

    Validation
    Type enforcement
    Constraints
    Defaults
    Python model

Then:

    PydanticOutputParser

---

# 45. Quick Decision Guide

    Plain text
        ↓
    StrOutputParser

    JSON
        ↓
    JsonOutputParser

    Simple predefined structure
        ↓
    StructuredOutputParser

    Structured + validation
        ↓
    PydanticOutputParser

---

# 46. Important Points to Remember

1. LLMs normally return natural language.
2. Applications often require structured data.
3. Output parsers convert LLM output into useful formats.
4. `StrOutputParser` is the simplest parser.
5. `StrOutputParser` returns a string.
6. `JsonOutputParser` is used for JSON output.
7. `StructuredOutputParser` works with predefined response schemas.
8. `ResponseSchema` defines expected fields for `StructuredOutputParser`.
9. `PydanticOutputParser` uses Pydantic models.
10. Pydantic provides validation and parsing.
11. `TypedDict` mainly provides type hints and dictionary structure.
12. TypedDict should not be confused with runtime validation.
13. JSON Schema is a standard way to describe JSON structure.
14. Pydantic is especially useful for Python applications.
15. JSON Schema is language-independent.
16. Output parsers are commonly used with LCEL.
17. A common chain pattern is:

        prompt | model | parser

18. `get_format_instructions()` is important for structured parsers.
19. Structured output makes LLM applications more reliable.
20. Structured output is especially useful in production systems.

---

# 47. Final Revision Summary

## Output Parsers

    Raw LLM Output
          ↓
    Output Parser
          ↓
    Structured / Usable Output

### StrOutputParser

    Purpose:
    Raw LLM response → String

    Use when:
    Only text is required.

---

### JsonOutputParser

    Purpose:
    LLM JSON → Parsed JSON / Dictionary

    Use when:
    JSON data is required.

---

### StructuredOutputParser

    Purpose:
    LLM → predefined fields

    Main concepts:
    ResponseSchema
    get_format_instructions()

    Use when:
    Simple structured output is required.

---

### PydanticOutputParser

    Purpose:
    LLM → validated Pydantic object

    Main concepts:
    BaseModel
    Field
    Validation
    Parsing
    get_format_instructions()

    Use when:
    Strict structured output and validation are required.

---

# 48. One-Line Memory Trick

    Str = String
    JSON = JSON
    Structured = Defined Fields
    Pydantic = Defined Fields + Validation

---

# 49. Overall Mental Model

    User
      ↓
    Prompt
      ↓
    LLM
      ↓
    Raw Response
      ↓
    Output Parser
      ↓
    ┌───────────────────────────────┐
    │ StrOutputParser               │ → String
    │ JsonOutputParser               │ → JSON / Dict
    │ StructuredOutputParser         │ → Structured Fields
    │ PydanticOutputParser           │ → Validated Object
    └───────────────────────────────┘
      ↓
    Application Logic
      ↓
    API / Database / UI / Automation

---

# 50. Exam / Interview Questions

### Q1. What is an Output Parser?

An Output Parser is a LangChain component that converts raw LLM responses into a structured or usable format.

### Q2. What does StrOutputParser do?

It converts the LLM response into a plain string.

### Q3. When should JsonOutputParser be used?

When the application needs JSON-structured output.

### Q4. What is StructuredOutputParser?

It is a parser that helps enforce a predefined set of response fields using response schemas.

### Q5. What is PydanticOutputParser?

It is a LangChain output parser that uses a Pydantic model to define and validate structured LLM output.

### Q6. What is the difference between TypedDict and Pydantic?

TypedDict mainly provides type hints and dictionary structure, whereas Pydantic provides runtime validation, parsing, constraints and model-based data handling.

### Q7. What is JSON Schema?

JSON Schema is a standard way of describing and validating the structure of JSON data.

### Q8. Which is better for strict validation?

Pydantic is generally the better choice when working in Python and requiring strong validation and parsing.

### Q9. What does `get_format_instructions()` do?

It generates formatting instructions that can be provided to the LLM so that the model knows the expected output structure.

### Q10. What is the common LangChain chain pattern for output parsing?

    prompt | model | parser

---

# 51. Final Cheat Sheet

| Requirement | Recommended Approach |
|---|---|
| Just text | StrOutputParser |
| JSON data | JsonOutputParser |
| Simple fixed fields | StructuredOutputParser |
| Strict validation | PydanticOutputParser |
| Type hints only | TypedDict |
| Python validation | Pydantic |
| Standard language-independent schema | JSON Schema |

## Remember

    StrOutputParser
        = String

    JsonOutputParser
        = JSON

    StructuredOutputParser
        = Predefined Structure

    PydanticOutputParser
        = Structure + Validation

    TypedDict
        = Type Hints

    Pydantic
        = Validation + Parsing

    JSON Schema
        = Standard JSON Structure